In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# --- Définition du modèle ---
class TinyCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 4, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(4, num_classes)
        )

    def forward(self, x):
        return self.model(x)

# --- Charger les poids depuis decrypted_avg.txt ---
def load_weights_from_txt(model, path):
    with open(path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    weights = {}
    for i in range(0, len(lines), 2):
        name = lines[i]
        values = list(map(float, lines[i + 1].split()))
        weights[name] = values

    with torch.no_grad():
        for name, param in model.named_parameters():
            if name not in weights:
                raise KeyError(f" Paramètre {name} introuvable dans {path}")
            flat_vals = torch.tensor(weights[name])
            param.copy_(flat_vals.view_as(param))

# --- Préparation des données ---
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

data_dir = "data_hopital2"  # test sur les données d'entraînement ou autre centre
dataset = datasets.ImageFolder(data_dir, transform=transform)
loader = DataLoader(dataset, batch_size=32, shuffle=False)
class_names = dataset.classes

# --- Chargement du modèle ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TinyCNN(num_classes=len(class_names)).to(device)

# --- Chargement des poids fédérés déchiffrés ---
load_weights_from_txt(model, "client2/decrypted_avg.txt")
model.eval()

# --- Évaluation du modèle ---
correct = 0
total = 0
with torch.no_grad():
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total if total > 0 else 0.0

# --- Affichage final ---
print(f"Classes : {class_names}")
print(f"Accuracy (modèle agrégé) : {accuracy:.4f}")


Classes : ['AbdomenCT', 'BreastMRI', 'ChestCT']
Accuracy (modèle agrégé) : 0.9413
